In [0]:
# ============================================================
# STEP 12A: LOAD SILVER SOURCE DATA
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window

EMPLOYEE_SOURCE = "workspace.silver.generated_employees"
COMPANY_SOURCE = "workspace.silver.generated_companies"

PREPARED_EMPLOYEE_TABLE = (
    "workspace.silver.prepared_employees"
)

# Load inference-facing datasets
employees_df = spark.table(EMPLOYEE_SOURCE)
companies_df = spark.table(COMPANY_SOURCE)

print("Silver source datasets loaded successfully.")

print("Total employees:", employees_df.count())
print("Total companies:", companies_df.count())

print("\nEmployee columns:")
print(employees_df.columns)

print("\nCompany columns:")
print(companies_df.columns)

In [0]:
# ============================================================
# STEP 12B: TITLE-BASED SENIORITY FEATURE ENGINEERING
# ============================================================

employees_df = employees_df.withColumn(
    "title_clean",
    F.lower(F.trim(F.col("job_title")))
)

employees_df = employees_df.withColumn(
    "seniority_signal",
    F.when(
        F.col("title_clean").rlike(r"\b(chief|ceo|president)\b"), "Executive"
    )
    .when(
        F.col("title_clean").rlike(r"\b(vice president|vp)\b"), "VP"
    )
    .when(
        F.col("title_clean").rlike(r"\b(director|head of)\b"), "Director"
    )
    .when(
        F.col("title_clean").rlike(r"\b(senior manager)\b"), "Senior Manager"
    )
    .when(
        F.col("title_clean").rlike(r"\b(manager)\b"), "Manager"
    )
    .when(
        F.col("title_clean").rlike(r"\b(team lead|technical lead|lead)\b"), "Team Lead"
    )
    .when(
        F.col("title_clean").rlike(r"\b(principal|staff|senior)\b"), "Senior IC"
    )
    .when(
        F.col("title_clean").rlike(r"\b(junior|associate)\b"), "Junior"
    )
    .when(
        F.col("title_clean").rlike(r"\b(intern|trainee)\b"), "Intern"
    )
    .otherwise("Individual Contributor")
)

# Lower number means more senior.
employees_df = employees_df.withColumn(
    "inferred_seniority_rank",
    F.when(F.col("seniority_signal") == "Executive", 0)
    .when(F.col("seniority_signal") == "VP", 1)
    .when(F.col("seniority_signal") == "Director", 2)
    .when(F.col("seniority_signal") == "Senior Manager", 3)
    .when(F.col("seniority_signal") == "Manager", 4)
    .when(F.col("seniority_signal") == "Team Lead", 5)
    .when(F.col("seniority_signal") == "Senior IC", 6)
    .when(F.col("seniority_signal") == "Individual Contributor", 7)
    .when(F.col("seniority_signal") == "Junior", 8)
    .when(F.col("seniority_signal") == "Intern", 9)
)

print("Title-based seniority features created.")

display(
    employees_df
    .groupBy("seniority_signal", "inferred_seniority_rank")
    .count()
    .orderBy("inferred_seniority_rank")
)

In [0]:
# ============================================================
# STEP 12B-12C: CEO-AWARE SENIORITY AND MANAGER ELIGIBILITY
# ============================================================

# Clean job titles
employees_df = employees_df.withColumn(
    "title_clean",
    F.lower(F.trim(F.col("job_title")))
)

# Classify seniority
employees_df = employees_df.withColumn(
    "seniority_signal",

    F.when(
        F.col("title_clean").rlike(
            r"\b(ceo|chief executive officer)\b"
        ), "CEO"
    )
    .when(
        F.col("title_clean").rlike(r"\b(chief|president)\b"),
        "Executive"
    )
    .when(
        F.col("title_clean").rlike(r"\b(vice president|vp)\b"),
        "VP"
    )
    .when(
        F.col("title_clean").rlike(r"\b(director|head of)\b"),
        "Director"
    )
    .when(
        F.col("title_clean").rlike(r"\bsenior\b.*\bmanager\b"),
        "Senior Manager"
    )
    .when(
        F.col("title_clean").rlike(r"\bmanager\b"),
        "Manager"
    )
    .when(
        F.col("title_clean").rlike(
            r"\b(team lead|technical lead|lead)\b"
        ), "Team Lead"
    )
    .when(
        F.col("title_clean").rlike(r"\b(principal|staff|senior)\b"),
        "Senior IC"
    )
    .when(
        F.col("title_clean").rlike(r"\b(junior|associate)\b"),
        "Junior"
    )
    .when(
        F.col("title_clean").rlike(r"\b(intern|trainee)\b"),
        "Intern"
    )
    .otherwise("Individual Contributor")
)

# Assign inferred seniority ranks
seniority_rank_map = {
    "CEO": 0,
    "Executive": 1,
    "VP": 2,
    "Director": 3,
    "Senior Manager": 4,
    "Manager": 5,
    "Team Lead": 6,
    "Senior IC": 7,
    "Individual Contributor": 8,
    "Junior": 9,
    "Intern": 10
}

employees_df = employees_df.withColumn(
    "inferred_seniority_rank",
    F.create_map(
        *[
            item
            for key, value in seniority_rank_map.items()
            for item in (F.lit(key), F.lit(value))
        ][0:]
    )[F.col("seniority_signal")]
)

# Identify potential managers
employees_df = employees_df.withColumn(
    "is_potential_manager",
    F.when(
        F.col("seniority_signal").isin(
            "CEO",
            "Executive",
            "VP",
            "Director",
            "Senior Manager",
            "Manager",
            "Team Lead"
        ),
        1
    ).otherwise(0)
)

# Display results
display(
    employees_df
    .groupBy(
        "seniority_signal",
        "inferred_seniority_rank",
        "is_potential_manager"
    )
    .count()
    .orderBy("inferred_seniority_rank")
)

print(
    "Potential manager candidates:",
    employees_df.filter(
        F.col("is_potential_manager") == 1
    ).count()
)

print(
    "CEO count:",
    employees_df.filter(
        F.col("seniority_signal") == "CEO"
    ).count()
)

In [0]:
from pyspark.sql import functions as F

employees_df = employees_df.withColumn(
    "title_clean",
    F.lower(F.trim(F.col("job_title")))
)

employees_df = employees_df.withColumn(
    "seniority_signal",
    F.when(
        F.col("title_clean").rlike(r"\b(ceo|chief executive officer)\b"), "CEO"
    )
    .when(
        F.col("title_clean").rlike(r"\b(vice president|vp)\b"), "VP"
    )
    .when(
        F.col("title_clean").rlike(r"\b(chief|president)\b"), "Executive"
    )
    .when(
        F.col("title_clean").rlike(r"\b(director|head of)\b"), "Director"
    )
    .when(
        F.col("title_clean").rlike(r"\bsenior\b.*\bmanager\b"), "Senior Manager"
    )
    .when(
        F.col("title_clean").rlike(r"\bmanager\b"), "Manager"
    )
    .when(
        F.col("title_clean").rlike(r"\b(team lead|technical lead|lead)\b"), "Team Lead"
    )
    .when(
        F.col("title_clean").rlike(r"\b(principal|staff|senior)\b"), "Senior IC"
    )
    .when(
        F.col("title_clean").rlike(r"\b(junior|associate)\b"), "Junior"
    )
    .when(
        F.col("title_clean").rlike(r"\b(intern|trainee)\b"), "Intern"
    )
    .otherwise("Individual Contributor")
)

seniority_rank_map = {
    "CEO": 0,
    "Executive": 1,
    "VP": 2,
    "Director": 3,
    "Senior Manager": 4,
    "Manager": 5,
    "Team Lead": 6,
    "Senior IC": 7,
    "Individual Contributor": 8,
    "Junior": 9,
    "Intern": 10
}

rank_expression = F.create_map(
    *[
        item
        for key, value in seniority_rank_map.items()
        for item in (F.lit(key), F.lit(value))
    ]
)

employees_df = employees_df.withColumn(
    "inferred_seniority_rank",
    rank_expression[F.col("seniority_signal")]
)

manager_categories = [
    "CEO", "Executive", "VP", "Director",
    "Senior Manager", "Manager", "Team Lead"
]

employees_df = employees_df.withColumn(
    "is_potential_manager",
    F.when(F.col("seniority_signal").isin(manager_categories), 1).otherwise(0)
)

display(
    employees_df
    .groupBy("seniority_signal", "inferred_seniority_rank", "is_potential_manager")
    .count()
    .orderBy("inferred_seniority_rank")
)

print("Total employees:", employees_df.count())
print(
    "CEO count:",
    employees_df.filter(F.col("seniority_signal") == "CEO").count()
)
print(
    "Potential manager candidates:",
    employees_df.filter(F.col("is_potential_manager") == 1).count()
)

In [0]:
from pyspark.sql import functions as F

# Start from the original Silver employee source.
base_employees_df = spark.table("workspace.silver.generated_employees")

# Rebuild title-based seniority classification.
prepared_base_df = base_employees_df.withColumn(
    "title_clean",
    F.lower(F.trim(F.col("job_title")))
)

prepared_base_df = prepared_base_df.withColumn(
    "seniority_signal",
    F.when(F.col("title_clean").rlike(r"\b(ceo|chief executive officer)\b"), "CEO")
    .when(F.col("title_clean").rlike(r"\b(vice president|vp)\b"), "VP")
    .when(F.col("title_clean").rlike(r"\b(chief|president)\b"), "Executive")
    .when(F.col("title_clean").rlike(r"\b(director|head of)\b"), "Director")
    .when(F.col("title_clean").rlike(r"\bsenior\b.*\bmanager\b"), "Senior Manager")
    .when(F.col("title_clean").rlike(r"\bmanager\b"), "Manager")
    .when(F.col("title_clean").rlike(r"\b(team lead|technical lead|lead)\b"), "Team Lead")
    .when(F.col("title_clean").rlike(r"\b(principal|staff|senior)\b"), "Senior IC")
    .when(F.col("title_clean").rlike(r"\b(junior|associate)\b"), "Junior")
    .when(F.col("title_clean").rlike(r"\b(intern|trainee)\b"), "Intern")
    .otherwise("Individual Contributor")
)

seniority_rank_map = {
    "CEO": 0,
    "Executive": 1,
    "VP": 2,
    "Director": 3,
    "Senior Manager": 4,
    "Manager": 5,
    "Team Lead": 6,
    "Senior IC": 7,
    "Individual Contributor": 8,
    "Junior": 9,
    "Intern": 10
}

rank_expression = F.create_map(
    *[
        item
        for key, value in seniority_rank_map.items()
        for item in (F.lit(key), F.lit(value))
    ]
)

prepared_base_df = prepared_base_df.withColumn(
    "inferred_seniority_rank",
    rank_expression[F.col("seniority_signal")]
)

manager_categories = [
    "CEO", "Executive", "VP", "Director",
    "Senior Manager", "Manager", "Team Lead"
]

prepared_base_df = prepared_base_df.withColumn(
    "is_potential_manager",
    F.when(F.col("seniority_signal").isin(manager_categories), 1).otherwise(0)
)

# Select company context. Do not expose organizational_structure.
company_context_df = spark.table(
    "workspace.silver.generated_companies"
).select(
    "company_id",
    "industry",
    "company_size",
    F.col("employee_count").alias("company_employee_count"),
    "country"
)

# Join employee features with company context.
prepared_employees_df = prepared_base_df.join(
    company_context_df,
    on="company_id",
    how="left"
).select(
    "employee_id",
    "company_id",
    "employee_name",
    "job_title",
    "department",
    "division",
    "industry",
    "company_size",
    "company_employee_count",
    "country",
    "title_clean",
    "seniority_signal",
    "inferred_seniority_rank",
    "is_potential_manager"
)

# Verify results.
display(prepared_employees_df.limit(10))

print("Prepared employee records:", prepared_employees_df.count())
print(
    "Distinct companies:",
    prepared_employees_df.select("company_id").distinct().count()
)
print("Prepared columns:", prepared_employees_df.columns)
print(
    "Organizational structure included:",
    "organizational_structure" in prepared_employees_df.columns
)

In [0]:
from pyspark.sql import functions as F

# Check seniority classification.
display(
    prepared_employees_df
    .groupBy("seniority_signal", "inferred_seniority_rank")
    .count()
    .orderBy("inferred_seniority_rank")
)

# Check company-level employee counts.
display(
    prepared_employees_df
    .groupBy("company_id")
    .agg(
        F.count("*").alias("employee_records"),
        F.countDistinct("employee_id").alias("unique_employees"),
        F.countDistinct("industry").alias("industry_values"),
        F.countDistinct("company_size").alias("company_size_values")
    )
    .orderBy("company_id")
    .limit(10)
)

# Check for missing company context.
print(
    "Employees missing industry:",
    prepared_employees_df.filter(F.col("industry").isNull()).count()
)

print(
    "Employees missing company size:",
    prepared_employees_df.filter(F.col("company_size").isNull()).count()
)

print(
    "Duplicate employee records:",
    prepared_employees_df.count()
    - prepared_employees_df.select("employee_id").distinct().count()
)

In [0]:
from pyspark.sql import functions as F

PREPARED_EMPLOYEE_TABLE = "workspace.silver.prepared_employees"

# Validate the prepared dataset before saving.
total_employees = prepared_employees_df.count()
unique_employees = prepared_employees_df.select("employee_id").distinct().count()
total_companies = prepared_employees_df.select("company_id").distinct().count()

missing_company_context = prepared_employees_df.filter(
    F.col("industry").isNull() |
    F.col("company_size").isNull() |
    F.col("company_employee_count").isNull()
).count()

duplicate_employees = total_employees - unique_employees

assert total_employees == 74610, f"Unexpected employee count: {total_employees}"
assert unique_employees == total_employees, "Duplicate employee IDs detected"
assert total_companies == 500, f"Unexpected company count: {total_companies}"
assert missing_company_context == 0, "Missing company context detected"
assert "organizational_structure" not in prepared_employees_df.columns
assert "true_manager_id" not in prepared_employees_df.columns

# Save the inference-ready employee dataset.
(
    prepared_employees_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PREPARED_EMPLOYEE_TABLE)
)

# Read back and verify the saved table.
saved_employees_df = spark.table(PREPARED_EMPLOYEE_TABLE)

print("Prepared employee table saved and read back.")
print("Table:", PREPARED_EMPLOYEE_TABLE)
print("Saved records:", saved_employees_df.count())
print("Unique employees:", saved_employees_df.select("employee_id").distinct().count())
print("Companies:", saved_employees_df.select("company_id").distinct().count())
print("Columns:", saved_employees_df.columns)